# 02 · Model Training

Trains and compares multiple regression models. This notebook calls the exact same functions used by `python -m src.train`, so results here match the production training script.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import config, data_loader, feature_engineering, train as train_module

## Load data and split (train / validation / test)

In [2]:
train_df, val_df, test_df = data_loader.load_and_split()
len(train_df), len(val_df), len(test_df)

(2051, 439, 440)

## Feature engineering

In [3]:
X_train, y_train = train_module._prepare_xy(train_df)
X_val, y_val = train_module._prepare_xy(val_df)
X_train.head()

,Lot Area,Overall Qual,Overall Cond,Gr Liv Area,Garage Cars,property_age,years_since_renovation,total_bathrooms,total_rooms,living_area_per_bedroom,lot_area_per_living_area,garage_indicator,has_fireplace,has_deck_or_porch,floors,Neighborhood,Bldg Type,Central Air
0,6270,5,6,2002,3.0,58,57,2.0,8,400.400000,3.131868,1,0,0,2.0,Crawfor,Duplex,N
1,25095,5,8,1473,1.0,41,6,1.0,5,736.500000,17.036660,1,1,1,1.0,ClearCr,1Fam,Y
2,8238,6,5,1525,2.0,9,8,2.5,6,381.250000,5.401967,1,1,1,2.0,Gilbert,1Fam,Y
3,3907,8,5,1191,2.0,21,21,2.0,5,397.000000,3.280437,1,1,1,1.0,Blueste,TwnhsE,Y
4,3072,7,5,1414,2.0,2,2,2.0,6,471.333333,2.172560,1,1,1,1.0,Blmngtn,TwnhsE,Y


## Train and compare baseline models

Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting, and XGBoost, each wrapped in the same preprocessing pipeline (`src/preprocessing.py`) to avoid preprocessing leakage.

In [4]:
comparison_df, fitted_pipelines = train_module.train_and_compare(
    X_train, y_train, X_val, y_val
)
comparison_df

,Model,Train_MAE,Train_RMSE,Train_R2,Val_MAE,Val_RMSE,Val_R2,Val_MAPE,Train_Time_Sec
0,XGBoost,425.70,663.43,0.9999,16679.98,24293.24,0.9138,9.77,0.35
1,Gradient Boosting,13995.40,19157.91,0.9397,16485.32,24456.15,0.9127,9.57,0.40
2,Random Forest,6573.67,10840.39,0.9807,16517.74,25306.66,0.9065,9.55,3.29
3,Linear Regression,19265.61,30195.41,0.8502,18829.96,28103.06,0.8847,10.65,0.03
4,Ridge Regression,19247.81,30215.61,0.8500,18840.81,28093.66,0.8847,10.67,0.01
5,Lasso Regression,19390.51,30597.88,0.8462,19123.83,28363.28,0.8825,10.93,0.02


## Hyperparameter tuning

The strongest tree-based model from the comparison above is tuned with `RandomizedSearchCV` (5-fold cross-validation on the training set only).

In [5]:
tree_models = comparison_df[comparison_df['Model'].isin(
    ['Random Forest', 'Gradient Boosting', 'XGBoost']
)]
best_tree_model_name = tree_models.iloc[0]['Model']
best_tree_model_name

'XGBoost'

In [6]:
best_estimator, best_params, best_cv_score = train_module.tune_best_tree_model(
    best_tree_model_name, X_train, y_train
)
best_params

{'model__subsample': 0.8,
 'model__n_estimators': 600,
 'model__max_depth': 3,
 'model__learning_rate': 0.05,
 'model__colsample_bytree': 0.9}

In [7]:
tuned_val_pred = best_estimator.predict(X_val)
from src import utils
utils.compute_regression_metrics(y_val, tuned_val_pred)

{'MAE': 15073.33,
 'MSE': 505852384.0,
 'RMSE': 22491.16,
 'R2': 0.9261,
 'MAPE': 8.81}

## Note

The full pipeline — including final model selection, retraining on train+validation, one-time test-set evaluation, and saving the model artifact — is orchestrated by `src/train.py`. Run it directly with:

```bash
python -m src.train
```